<img src='http://hilpisch.com/tpq_logo.png' width="350px" align="right">

# Certificate in Python for Algorithmic Trading

## Vectorized & Event-Based Backtesting

Dr. Yves J. Hilpisch

The Python Quants GmbH

http://tpq.io | <training@tpq.io>

<img src="http://hilpisch.com/images/tpq_bootcamp.png" width="350px" align="left">

## Strategy Idea

This section describes the strategy idea in the form of

* the hypothesis
* the reasoning behind it
* a success defition
* an example specification
* associated decision/trading rules

### Hypothesis

The hypothesis is as follows:

> Consider a financial instrument $S$ represented by a time series of price bars with homogenous length $b$ (e.g. 5 minutes). Let $SMA$ be a simple moving average over a number $x$  of bars (e.g. 15) of the time series and let $EWMA$ be an exponentially weighted moving average with some parameter value $\alpha$ (with e.g. $\alpha=0.2$). $\alpha$ is called the decay factor. The trading strategy that goes long $S$ when $SMA > EWMA$ and short when $SMA < EWMA$ is consistently profitable after transaction costs.

### Reasoning

The $SMA$ represents an approximation for the recent market environment/regime with regard to $S$ for which every included price level is weighted equally. $EWMA$ is an approximation for the overall market environment/regime &mdash; giving more or less weight to more recent price levels depending on the parameter $\alpha$. The idea is that if the $SMA$ sees a higher price level as justified than the $EWMA$ ($SMA > EWMA$), the price level is expected to further go up (on average). The strategy assumes a _positive momentum_. The opposite holds true for $SMA < EWMA$.

### Success Definition

The strategy is successful if

* the average (leveraged) return after transaction costs is positive and
* if it shows a Sharpe ratio of 1.5 or better after transaction costs.

### Example Specification

The following provides an example specification for a trading strategy based on the idea:<br><br>

| feature | description |
|------|-----|
| trading platform | Oanda |
| instrument |EUR_GBP |
| bar length | 10 minutes |
| SMA | 46 |
| EWMA (alpha) | 0.03 |
| leverage | none |

### Decision Rules

The following provides rules associated with the strategy idea:<br><br>

| action | description |
|------|-----|
| enter long position | as soon as $SMA > EWMA$ |
| enter short position | as soon as $SMA < EWMA$ |
| stop loss | as soon as loss on equity >= 2% |

## Strategy Analysis

This section analyzes the strategy idea addressing the following topics:

* data on which to base the analysis
* derivation of the indicator values
* derived data (e.g. log returns, positions)
* hit ratio of the strategy

To get started, some imports and configurations.

For the following, you need to install `tpqoa` via

    pip install git+git://github.com/yhilpisch/tpqoa

In [ ]:
 !git clone https://github.com/yhilpisch/tpqoa.git

In [ ]:
!pip install v20

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_addon.git
import sys
sys.path.append('python_for_algo_trading_addon')


In [ ]:
from tpqoa import tpqoa
import numpy as np
import pandas as pd
from pylab import plt
plt.style.use('seaborn-v0_8')
%matplotlib inline

### The Data

The following retrieves data according to the example specification.

In [ ]:
api = tpqoa.tpqoa('/content/python_for_algo_trading_addon/pyalgo.cfg')

In [ ]:
# api.get_instruments()

In [ ]:
instrument = 'EUR_GBP'
start = '2018-08-01'
end = '2018-10-01'
granularity = 'M10'

In [ ]:
fn = '/content/python_for_algo_trading_addon/oanda_{}_{}_{}_{}.csv'.format(instrument, start, end, granularity)
fn

In [ ]:
try:
    raw = pd.read_csv(fn, index_col=0, parse_dates=True)
except:
    raw_a = api.get_history(instrument, start, end, granularity, 'A')
    raw_b = api.get_history(instrument, start, end, granularity, 'B')
    raw = pd.DataFrame({'ask': raw_a['c'], 'bid': raw_b['c']})
    raw.to_csv(fn)
raw['mid'] = raw.mean(axis=1)

In [ ]:
raw.head()

In [ ]:
raw['mid'].plot(figsize=(10, 6));

In [ ]:
(raw['ask'] - raw['bid']).head(10)

In [ ]:
spread = (raw['ask'] - raw['bid']).mean()
spread

In [ ]:
ptc = spread / raw['mid'].mean()
ptc

In [ ]:
import math

In [ ]:
ptc = math.log(raw['ask'].mean() / raw['bid'].mean())

In [ ]:
ptc

### The Indicators

The following calculates and visualizes the two indicators.

In [ ]:
# example parameters
SMA = 46
alpha = 0.03

In [ ]:
# parameters from optimization
SMA = 50
alpha = 0.029875

In [ ]:
data = pd.DataFrame(raw['mid'])
data['SMA'] = data['mid'].rolling(SMA).mean()
data['EWMA'] = data['mid'].ewm(alpha=alpha).mean()

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
data[['mid', 'SMA', 'EWMA']].iloc[-650:-250].plot(figsize=(10, 6), grid=True,
                                           style=['b', 'r--', 'y--']);

### Derived Data

We add the log returns to the data.

In [ ]:
data['returns'] = np.log(data['mid'] / data['mid'].shift())
data.dropna(inplace=True)

Adding the direction of the movement.

In [ ]:
data['direction'] = np.sign(data['returns'])

Next, the position according to the strategy idea.

In [ ]:
data['position'] = np.where(data['SMA'] > data['EWMA'], 1, -1)
data['position'] = data['position'].shift(1)
data.dropna(inplace=True)

In [ ]:
data.head()

### Hit Ratio

The hit ratio for the strategy.

In [ ]:
vc  = (data['position'] == data['direction']).value_counts()
vc

In [ ]:
vc[True] / vc.sum()

## Frequency Analysis

This section analyzes the following:

* statistics with regard to directional movements
* statistics with regard to log returns

### Directional Analysis

Via grouping, the frequency for directional movements &mdash; given the strategy position &mdash; is derived.

In [ ]:
grouped = data.groupby(['position', 'direction'])

In [ ]:
grouped['direction'].count() / len(data)

### Returns Analysis

Again via grouping, return statistics are derived given the strategy positions and the directional movement.

In [ ]:
data['strategy'] = data['returns'] * data['position']

In [ ]:
grouped['strategy'].aggregate([np.mean, np.median, np.std, len])

## Vectorized Backtesting

This section implements a vectorized backtesting of the example specification. It analyzes

* the number of trades placed
* the performance before transaction costs
* the performance after transaction costs
* the maximum drawdown observed and when
* the Sharpe ratio before and after transaction costs

### Number of Trades

Changes in the strategy position values indicate a trade.

In [ ]:
(data['position'].diff() != 0).value_counts()

### Performance (Before Transaction Costs)

The performance of the strategy over the whole time period (unleveraged, before transaction costs) compared to the passive benchmark investment.

In [ ]:
data[['returns', 'strategy']].sum().apply(np.exp)

The following shows the performance of the strategy (unleveraged, before transaction costs) compared to the passive benchmark investment over time.

In [ ]:
data[['returns', 'strategy']].cumsum().apply(np.exp).plot(
        figsize=(10, 6), grid=True, style=['b', 'r']);

### Performance (After Transaction Costs)

For every trade the proportional transaction costs are deducted from the strategy performance before transaction costs (at the point in time when the trade happens).

In [ ]:
data['strategy_tc'] = np.where(data['position'].diff() != 0,
                        data['strategy'] - ptc, data['strategy'])

The performance of the strategy over the whole time period (unleveraged, after transaction costs) compared to the passive benchmark investment.

In [ ]:
data[['returns', 'strategy_tc']].sum().apply(np.exp)

 The following shows the performance of the strategy (unleveraged, before and after transaction costs) compared to the passive benchmark investment over time.

In [ ]:
data[['returns', 'strategy', 'strategy_tc']].cumsum().apply(np.exp).plot(
    figsize=(10, 6), grid=True, style=['b', 'r', 'm']);

### Drawdown

The following derives the drawdown values, the maximum drawdown and the point in time when it happens.

In [ ]:
data['drawdown'] = (data['strategy_tc'].cumsum().cummax().apply(np.exp) -
                     data['strategy_tc'].cumsum().apply(np.exp))

Drawdown in currency units (price).

In [ ]:
mdd = data['drawdown'].max()
mdd

In [ ]:
ddt = data['drawdown'].idxmax()
ddt

Drawdown in per cent.

In [ ]:
mdd / data['strategy_tc'].cumsum().cummax().apply(np.exp).loc[ddt]

Below a visualization of the strategy performance over time, the maxmimum values reached over time and the maximum drawdown event.

In [ ]:
ax = data['strategy_tc'].cumsum().apply(np.exp).plot(style=['b'])
data['strategy_tc'].cumsum().cummax().apply(np.exp).plot(
    figsize=(10, 6), grid=True, style=['r'], ax=ax)
plt.axvline(ddt, color='y', lw=2.0, ls='--');

### Sharpe Ratio

The Sharpe ratio of the strategy before transaction costs is:

In [ ]:
data['strategy'].mean() * len(data) ** 0.5 / data['strategy'].std()

The Sharpe ratio of the strategy after transaction costs is:

In [ ]:
data['strategy_tc'].mean() * len(data) ** 0.5 / data['strategy_tc'].std()

## Optimization

### Backtesting Function

First, the function to implement the backtesting given a certain data (sub-) set and the two major strategy parameters.

In [ ]:
import math

In [ ]:
def backtest_strategy(data, SMA, alpha, leverage=1):
    df = data.copy()
    df['SMA'] = df['mid'].rolling(SMA).mean()
    df['EWMA'] = df['mid'].ewm(alpha=alpha).mean()
    df['returns'] = np.log(df['mid'] / df['mid'].shift())
    df['direction'] = np.sign(df['returns'])
    df['position'] = np.where(df['SMA'] > df['EWMA'], 1, -1)
    df['position'] = df['position'].shift(1)
    df.dropna(inplace=True)
    df['strategy'] = df['returns'] * df['position'] * leverage
    df['strategy_tc'] = np.where(df['position'].diff() != 0,
                                 df['strategy'] - ptc * leverage,
                                 df['strategy'])
    vc  = (df['position'] == df['direction']).value_counts()
    srb = df['strategy'].mean() * len(df) ** 0.5 / df['strategy'].std()
    sra = df['strategy_tc'].mean() * len(df) ** 0.5 / df['strategy_tc'].std()
    df['drawdown'] = (df['strategy_tc'].cumsum().cummax().apply(np.exp) -
                      df['strategy_tc'].cumsum().apply(np.exp))
    mdda = df['drawdown'].max()
    ddt = df['drawdown'].idxmax()
    mddr = mdda / df['strategy_tc'].cumsum().cummax().apply(np.exp).loc[ddt]
    try:
        trades = int((df['position'].diff() != 0).value_counts()[True])
    except:
        trades = 0
    res = pd.DataFrame({'benchmark': math.exp(df['returns'].sum()),
                        'SMA': SMA, 'alpha': alpha,
                        'hit ratio': vc[True] / vc.sum(),
                        'strategy': math.exp(df['strategy'].sum()),
                        'Sharpe': srb,
                        'strategy_tc': math.exp(df['strategy_tc'].sum()),
                        'Sharpe_tc': sra,
                        'max dd [$]': mdda, 'max dd [%]': mddr,
                        'trades': trades
                       },
                      index=[0])
    return res[['SMA', 'alpha', 'hit ratio', 'benchmark', 'strategy', 'Sharpe',
                'strategy_tc', 'Sharpe_tc', 'max dd [$]',
                'max dd [%]', 'trades']]

In [ ]:
backtest_strategy(pd.DataFrame(raw['mid']), 50, 0.03, 5)

### Training

Second, the optimization procedure to derive optimal parameters given the training data sub-set.

In [ ]:
from itertools import product

In [ ]:
split = int(0.7 * len(raw))

In [ ]:
train = raw.iloc[:split].copy()

In [ ]:
train.info()

In [ ]:
SMAs = range(10, 61, 5)
alphas = np.linspace(0.001, 0.1, 25)

In [ ]:
%%time
res = pd.DataFrame()
for SMA, alpha in product(SMAs, alphas):
    res = res.append(backtest_strategy(train, SMA, alpha, 10),
                     ignore_index=True)

In [ ]:
res.info()

In [ ]:
res.sort_values('strategy_tc', ascending=False).head(10)

## Testing

Third, the testing of the optimal parameters given the testing data sub-set.

In [ ]:
test = raw.iloc[split:].copy()

In [ ]:
opt = res.iloc[res['strategy_tc'].idxmax()]
opt[['SMA', 'alpha']]

In [ ]:
backtest_strategy(test, int(opt['SMA']), opt['alpha'], 10)

## Event-based Backtesting

### Adding Stop Loss

**REMARK**: Some issues (re performance numbers) that arose during the live session seem to be due to data issues when the end date is increased to beyond 1. October 2018.

In [ ]:
%run sma_ewma_backtest.py

In [ ]:
instrument = 'EUR_GBP'
start = '2018-08-01'
end = '2018-10-28'
granularity = 'M10'

In [ ]:
sma = SmaEwmaBacktester(instrument, start, end, granularity, 10000, verbose=False)

In [ ]:
sma.data['mid'].plot();

In [ ]:
sma.data[(sma.data.index >= '2018-10-09') &
        (sma.data.index <= '2018-10-10')]['mid'].plot();

In [ ]:
sma.data[(sma.data.index >= '2018-10-25') &
        (sma.data.index <= '2018-10-26')]['mid'].plot();

In [ ]:
sma.run_strategy(int(opt['SMA']), opt['alpha'], 10000, 20, 0.10, 5)

### Adding Trading Times

In [ ]:
%run sma_ewma_backtest_window.py

In [ ]:
start_time = 5  # hour
end_time = 17  # hour

In [ ]:
sma = SmaEwmaBacktesterWindow('EUR_GBP', start, end, granularity, 10000, verbose=False)

In [ ]:
sma.run_strategy(int(opt['SMA']), opt['alpha'], 10000, 10, 0.15, 5, start_time, end_time)

<img src='http://hilpisch.com/tpq_logo.png' width="350px" align="right">